# 🧪 Provider Execution + Log Validation

### 🔁 Step 1: Reset DB and Seed All Providers

In [1]:
import os
from pathlib import Path
import sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from app.db.base import Base
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers

close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(bind=engine) as session:
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_context_providers(session)
    seed_prompt_providers(session)
    seed_score_providers(session)
    seed_tool_providers(session)
print("✅ All providers seeded.")

Working directory is now: C:\Repos\codecritic
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
Seeded context provider configurations successfully.
Seeded prompt provider configurations successfully.
Seeded score providers successfully.
Seeded tool configurations successfully.
✅ All providers seeded.


### 🔍 Step 2: Load Agent Provider via Factory

In [2]:
from sqlalchemy import select
from app.db.models import AgentProviderConfig
from app.factories.agent_provider_factory import AgentProviderFactory

with Session(bind=engine) as session:
    record = session.execute(
        select(AgentProviderConfig).where(AgentProviderConfig.name == "Basic Agent Provider")
    ).scalar_one()
    provider = AgentProviderFactory.create(record.id)
print("✅ AgentProvider loaded:", provider.__class__.__name__)

✅ AgentProvider loaded: BasicAgentProvider


### 🏃 Step 3: Run Provider and Log Output

In [3]:
session_id = "test-session-001"
response = provider.run(input={"goal": "test action"}, session_id=session_id)
print("✅ AgentProvider response:", response)

✅ AgentProvider response: [Agent] Executed with config: {'goal': 'test action'}


### 📜 Step 4: Query Provider Log

In [4]:
from sqlalchemy import text

with engine.connect() as conn:
    logs = conn.execute(
        text("SELECT * FROM provider_log WHERE session_id = 'test-session-001'")
    ).fetchall()

assert logs, "❌ No provider logs found for session!"
print("✅ Provider Log Entries:")
for row in logs:
    print(row)

✅ Provider Log Entries:
(1, 'test-session-001', '2025-05-25T02:17:21.589896+00:00', 1, 'AgentProviderBase', '{"goal": "test action"}', '"[Agent] Executed with config: {\'goal\': \'test action\'}"', None, None, None, None)
